In [1]:
using Clapeyron, Metaheuristics, Printf

In [2]:
like_parameter = """Clapeyron Database File
PCSAFT Like Parameters [csvtype = like,grouptype = PCSAFT]
species,Mw,segment,sigma,epsilon,n_H,n_e
ethanolamine,61.08,3.73078,2.780462135,145.4800176,2,2
"""
assoc_parameter = """Clapeyron Database File
PCSAFT Assoc Parameters [csvtype = assoc,grouptype = PCSAFT]
species1,site1,species2,site2,epsilon_assoc,bondvol
ethanolamine,H,ethanolamine,e,2000.0,0.03
"""


model = PCSAFT(["ethanolamine"], userlocations = [like_parameter, assoc_parameter])
print(model.params.epsilon_assoc.values)

Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[2000.0]

In [3]:
# Run this ONCE to fix your CSV files
function fix_line_endings(filename)
    content = read(filename, String)
    fixed = replace(content, "\r\n" => "\n")
    write(filename, fixed)
    println("Fixed: $filename")
end

fix_line_endings("satp_ethanolamine.csv")
fix_line_endings("rhol_ethanolamine.csv")

Fixed: satp_ethanolamine.csv
Fixed: rhol_ethanolamine.csv


In [4]:
### Saturation pressure — output in Pa (SI)
function my_saturation_p(model::EoSModel, T::Float64)
    sat = saturation_pressure(model, T)
    return sat[1]   # Pa
end

# Liquid density — output in kg/m³
function my_liquid_density(model::EoSModel, T::Float64)
    sat  = saturation_pressure(model, T)
    Mw   = model.params.Mw[1]      # g/mol
    rhol = 1.0 / sat[2]            # mol/m³
    return rhol * Mw / 1000.0      # mol/m³ × g/mol ÷ 1000 = kg/m³
end

println(my_liquid_density(model, 293.15))
println(my_saturation_p(model, 273.15))

990.5222349797175
907.2188539878803


In [5]:
toestimate = [
    Dict(
        :param   => :epsilon_assoc,
        :indices => (1,1),
        :lower   => 0.0,
        :upper   => 5000.0,
        :guess   => 2400.0
    ),
    Dict(
        :param   => :bondvol,
        :indices => (1,1),
        :lower   => 0.0,
        :upper   => 1.0,
        :guess   => 0.4
    )
]

2-element Vector{Dict{Symbol, Any}}:
 Dict(:upper => 5000.0, :param => :epsilon_assoc, :indices => (1, 1), :guess => 2400.0, :lower => 0.0)
 Dict(:upper => 1.0, :param => :bondvol, :indices => (1, 1), :guess => 0.4, :lower => 0.0)

In [6]:
estimator, objective, x0, upper, lower = Estimation(
    model,
    toestimate,
    [
        "rhol_ethanolamine.csv",
        "satp_ethanolamine.csv",
    ]
)
 
println("Initial objective value: ", objective(x0))

Initial objective value: 0.8542847148637491


In [7]:
method = ECA(; options = Options(iterations = 10000000, seed = 999))
 
params_opt, model_opt = optimize(objective, estimator, method)

([2359.063780927415, 0.11665871424520459], PCSAFT{BasicIdeal, Float64}("ethanolamine"))

In [8]:
using CSV, DataFrames, Printf

function calculate_AAD(model, csv_file, property_func)
    df = CSV.read(csv_file, DataFrame, comment="#", skipto=4)
    
    input_col  = names(df)[1]          # first column = input (T)
    output_col = names(df)[2]          # second column = out_xxx (experimental)
    
    inputs   = df[!, input_col]
    exp_vals = df[!, output_col]
    
    println("\n=== AAD: $csv_file ===")
    @printf("%-10s  %-12s  %-12s  %-8s\n", input_col, "exp", "calc", "ARD%")
    
    errors = Float64[]
    for (i, x) in enumerate(inputs)
        calc = property_func(model, x)
        err  = abs(calc - exp_vals[i]) / abs(exp_vals[i]) * 100
        push!(errors, err)
        @printf("%-10.4f  %-12.6f  %-12.6f  %-8.4f\n", x, exp_vals[i], calc, err)
    end
    
    aard = sum(errors) / length(errors)
    @printf("AARD = %.4f%%\n", aard)
    return aard
end

calculate_AAD (generic function with 1 method)

In [9]:
aard_p   = calculate_AAD(model_opt, "satp_ethanolamine.csv",   my_saturation_p)


=== AAD: satp_ethanolamine.csv ===

┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593



Clapeyron Estimator  exp           calc          ARD%    
293.1500    38.200000     38.180495     0.0511  
303.1500    86.700000     86.640009     0.0692  
313.1500    185.500000    185.445012    0.0296  
323.1500    376.200000    376.465855    0.0707  
333.1500    727.400000    728.379311    0.1346  
343.1500    1347.200000   1348.888433   0.1253  
353.1500    2399.800000   2400.149094   0.0145  
363.1500    4126.300000   4117.402154   0.2156  
AARD = 0.0888%


0.08883868393470354

In [10]:
aard_p   = calculate_AAD(model_opt, "rhol_ethanolamine.csv",   my_liquid_density)


=== AAD: rhol_ethanolamine.csv ===
Clapeyron Estimator  exp           calc          ARD%    
303.1500    1009.200000   1001.885611   0.7248  
308.1500    1004.800000   997.628295    0.7137  
313.1500    1000.900000   993.343859    0.7549  
318.1500    996.900000    989.028882    0.7896  
323.1500    992.800000    984.680034    0.8179  
AARD = 0.7602%


┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593


0.7601789985631655